# Python Source Map: Alternative-Proposal Detection & Timing

For each decision episode (`Episode #`), this notebook reads every comment
already stored in `consensus.python_source_map_comments` (collected by the
companion ingestion notebook) and writes two derived fields back onto that
episode's document in `consensus.python_source_map`:

1. **`Alternative Comment Count`** — how many comments in the episode (not
   counting the original proposal itself) read as proposing an alternative,
   by a simple, fully-transparent keyword rule.
2. **`Latest Alternative Timing (% Elapsed)`** — how far into the episode's
   total timeline (from its first comment to its last) the *latest* such
   alternative appeared, as a percentage.

This notebook is **read-only** with respect to `python_source_map` -- it reads
`Episode #`s implicitly via the comments collection, computes both values, and
prints/returns them (as `results` and `results_df`) for you to inspect, export,
or write back yourself in a separate step. Nothing is written to MongoDB.


## Cell 1: Configuration and MongoDB connection

In [ ]:
import json
import time
import re
import requests
from pymongo import MongoClient, UpdateOne

with open("config.json") as f:
    config = json.load(f)

GITHUB_TOKEN = config.get("github_token")
mongo_uri = config["mongo_uri"]

client = MongoClient(mongo_uri)
db = client["consensus"]
source_map_coll = db["python_source_map"]
comments_coll = db["python_source_map_comments"]

print("Connected to MongoDB database 'consensus'.")
print(f"  - {source_map_coll.name}: {source_map_coll.estimated_document_count()} existing documents")
print(f"  - {comments_coll.name}: {comments_coll.estimated_document_count()} existing documents")


## Cell 2: The alternative-detection rule

**The whole algorithm, in one sentence:** a comment counts as proposing an
alternative if its text contains any of a fixed list of keywords/phrases,
checked case-insensitively — nothing more. No scoring, no machine learning,
no weighting; a reviewer can check any single classification just by
Ctrl-F'ing the comment text for the matched phrase.

The phrase list covers the two cases from the spec:
- An author explicitly flags *their own* comment as an alternative
  (`"alternative"`, `"alternatively"`, `"another approach"`, `"instead of"`, ...).
- Language pointing at a rejected/alternative idea, the way a PEP's own
  "Rejected Ideas" section would (`"rejected idea(s)"`, `"ideas rejected"`).

See `ALTERNATIVE_KEYWORDS` below for the exact list — edit it directly if
you want to add or remove a phrase; everything downstream just uses it.


In [ ]:
from datetime import datetime, timezone

# ---------------------------------------------------------------------------
# Alternative-detection: a single, transparent, fully-explainable rule.
#
# A comment is counted as an "alternative" if its text contains ANY of the
# phrases below, case-insensitively. That's the entire algorithm -- no
# machine learning, no scoring, just plain substring matching, so the
# classification of any individual comment can be checked by eye.
#
# The phrase list covers two things, per spec:
#   1. An author explicitly flagging their own comment as an alternative
#      ("alternative", "alternatively", "another approach", "instead of"...).
#   2. Language that points back at a rejected/alternative idea, the way a
#      PEP's own "Rejected Ideas" section would ("rejected idea(s)",
#      "ideas rejected", "alternative(s) considered").
# ---------------------------------------------------------------------------

ALTERNATIVE_KEYWORDS = [
    "alternative",       # covers "alternative", "alternatives", "alternative idea(s)/proposal/approach"
    "alternatively",
    "another approach",
    "another option",
    "another idea",
    "another proposal",
    "counter-proposal",
    "counterproposal",
    "instead of",
    "rejected idea",
    "ideas rejected",
    "idea rejected",
]


def is_alternative_comment(text):
    """Return True if `text` contains any alternative-signaling keyword,
    case-insensitively. This is the entire classification rule."""
    if not text:
        return False
    lowered = text.lower()
    return any(kw in lowered for kw in ALTERNATIVE_KEYWORDS)


def parse_dt(value):
    """Parse a Mongo-stored timestamp (ISO-8601 string, e.g.
    '2024-01-02T18:08:17Z') into a timezone-aware UTC datetime. Returns
    None if the value is missing or unparseable."""
    if not value:
        return None
    if isinstance(value, datetime):
        return value if value.tzinfo else value.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    except ValueError:
        return None


## Cell 3: Per-episode aggregation

For each `Episode #`:

1. Gather every comment belonging to it (across *all* of its threads, if it
   spanned more than one — see `thread_index`/`thread_count` on each comment).
2. Sort them by `created_at`. The **earliest** comment in the episode is
   treated as the original proposal and is excluded from the alternative
   count — by definition, the original proposal isn't an alternative to
   itself, regardless of what its own text happens to say.
3. Run the Cell 2 keyword rule over every *other* comment's `text`.
   `Alternative Comment Count` = however many come back positive. Python
   does all of the counting — the keyword rule is the only judgment call.
4. If there's at least one alternative, find the **latest** one by
   timestamp and compute:

   ```
   Latest Alternative Timing (% Elapsed) =
       (latest_alternative_time − episode_start_time)
       ────────────────────────────────────────────── × 100
       (episode_end_time − episode_start_time)
   ```

   where `episode_start_time`/`episode_end_time` are the earliest/latest
   comment timestamps in the whole episode. Worked example from spec: a
   thread starting 2026-09-01 12:00, ending 2026-10-03 13:00, with its
   latest alternative at 2026-10-02 12:30 → **96.81%** — reproduced exactly
   by this formula.

   If an episode has zero qualifying alternatives, this field is `None`
   (there's nothing to time). If every comment in an episode landed at the
   same timestamp (zero elapsed duration), it's also `None`, since a
   percentage of zero elapsed time isn't a meaningful quantity.

Each episode's two new fields are written with `update_one` (not an
upsert) against the existing `python_source_map` document for that
`Episode #` — if a comment's `Episode #` has no matching source-map row,
that's surfaced as an explicit warning rather than silently creating a
partial document.


In [ ]:
from collections import defaultdict
import pandas as pd

EPISODE_FIELD = "Episode #"

# ---------------------------------------------------------------------------
# Group every comment by its Episode # (decision episode / discussion
# thread -- across all of that episode's threads, if it had more than one).
# This cell only READS from python_source_map_comments; it does not write
# anything back to MongoDB. Results are returned/printed for inspection.
# ---------------------------------------------------------------------------

all_comments = list(comments_coll.find({}))
by_episode = defaultdict(list)
for c in all_comments:
    ep = c.get(EPISODE_FIELD)
    if ep is not None:
        by_episode[ep].append(c)

print(f"Loaded {len(all_comments)} comments across {len(by_episode)} episodes.")

results = []
no_timestamp_episodes = []

for episode, comments in by_episode.items():
    # Keep only comments we can actually place in time; one with no
    # created_at can't participate in "earliest/latest" reasoning.
    timed = [(parse_dt(c.get("created_at")), c) for c in comments]
    timed = [(dt, c) for dt, c in timed if dt is not None]

    if not timed:
        no_timestamp_episodes.append(episode)
        continue

    timed.sort(key=lambda x: x[0])

    thread_start = timed[0][0]
    thread_end = timed[-1][0]

    # The single earliest comment in the episode is treated as the original
    # proposal and is excluded from the alternative count -- it can't be an
    # "alternative" to itself, however its own text reads.
    original_proposal_id = timed[0][1].get("comment_id")

    alternative_timestamps = [
        dt for dt, c in timed
        if c.get("comment_id") != original_proposal_id and is_alternative_comment(c.get("text"))
    ]

    alternative_count = len(alternative_timestamps)

    if alternative_count == 0:
        latest_alternative_pct = None
    else:
        latest_alt_time = max(alternative_timestamps)
        duration = (thread_end - thread_start).total_seconds()
        if duration <= 0:
            # Every comment landed at (essentially) the same timestamp --
            # there's no meaningful "elapsed time" to express as a percentage.
            latest_alternative_pct = None
        else:
            elapsed = (latest_alt_time - thread_start).total_seconds()
            latest_alternative_pct = round(elapsed / duration * 100, 2)

    results.append({
        EPISODE_FIELD: episode,
        "Alternative Comment Count": alternative_count,
        "Latest Alternative Timing (% Elapsed)": latest_alternative_pct,
    })

results.sort(key=lambda r: r[EPISODE_FIELD])
results_df = pd.DataFrame(results)

print(f"\nComputed results for {len(results)} episode(s) "
      f"(not written back to MongoDB -- read-only pass).")
if no_timestamp_episodes:
    print(f"Skipped {len(no_timestamp_episodes)} episode(s) with no timestamped comments: {no_timestamp_episodes}")

print()
print(results_df.to_string(index=False))


## Notes / limitations

- **Keyword rule is intentionally simple**: it will miss alternatives
  phrased without any of the listed words (e.g. "What if we just used a
  decorator?" with no explicit "alternative"/"instead" framing), and it
  will occasionally flag a comment that uses one of the phrases without
  actually proposing anything (e.g. "that's not really an alternative,
  it's the same thing"). This is the tradeoff for a rule simple enough
  that every classification can be checked by eye — extend
  `ALTERNATIVE_KEYWORDS` in Cell 2 if you find systematic misses.
- **"Original proposal" = earliest timestamped comment in the episode**,
  full stop — not specifically `thread_index == 0` or a GitHub issue body
  vs. a Discourse first post. This keeps the rule source-agnostic and easy
  to state, and in practice the earliest comment *is* the issue body / the
  first Discourse post in every episode we've seen so far.
- **Timestamps are all treated as UTC** (Discourse and GitHub both report
  `created_at` in UTC already, e.g. `...Z` suffix).
- **Read-only**: this notebook never writes to `python_source_map` (or any
  other collection) -- it only reads `python_source_map_comments` and prints
  the computed values. Re-running it after the comments collection picks up
  new replies simply reprints refreshed numbers; to persist them, write
  `results`/`results_df` back to MongoDB yourself in a separate cell.
